
# GemmaFit E2B Local Video / Realtime Capability Harness

Purpose: profile the local E2B model before committing to a product architecture.

This notebook runs locally, prefers CUDA/GPU, auto-discovers test videos, samples frames, runs several prompt modes, stores raw outputs, and scores whether the response is usable for GemmaFit.

It intentionally tests failure modes:

- raw video freeform drift
- strict JSON compliance
- function-call style compliance
- unsupported medical / force / EMG / heart-rate claims
- latency for batch video and realtime sliding-window simulation

Product boundary: this is a profiling harness. It is not the production biomechanics judge. Production should still use pose evidence, capability contracts, deterministic gates, validators, and fallback.


In [ ]:

# 0. Local config
from __future__ import annotations

import json
import os
import re
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

# Resolve repo root whether the notebook is launched from repo root or finetune/.
ROOT = Path.cwd().resolve()
if not (ROOT / "implementation_plan.md").exists():
    if (ROOT.parent / "implementation_plan.md").exists():
        ROOT = ROOT.parent.resolve()
    else:
        raise RuntimeError("Run this notebook from D:/GemmaFit or D:/GemmaFit/finetune")

OUTPUT_DIR = ROOT / "test_assets" / "outputs" / "e2b_video_capability"
CONTACT_SHEET_DIR = OUTPUT_DIR / "contact_sheets"
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw_outputs"
for path in [OUTPUT_DIR, CONTACT_SHEET_DIR, RAW_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Model config.
# For true video/image understanding, point this at a Transformers-compatible multimodal Gemma model.
# Example override before launching Jupyter:
#   set GEMMA_E2B_MODEL=google/gemma-4-E2B-it
MODEL_ID_OR_PATH = os.environ.get("GEMMA_E2B_MODEL", "google/gemma-4-E2B-it")
BACKEND = os.environ.get("GEMMA_E2B_BACKEND", "transformers_multimodal")
TRUST_REMOTE_CODE = os.environ.get("TRUST_REMOTE_CODE", "0") == "1"
REQUIRE_GPU = os.environ.get("REQUIRE_GPU", "0") == "1"
LOAD_IN_4BIT = os.environ.get("LOAD_IN_4BIT", "auto").lower()  # "auto" | "1" | "0"

# Keep defaults small enough for local iteration. Increase after the first pass is stable.
MAX_VIDEOS = int(os.environ.get("MAX_VIDEOS", "8"))
FRAME_COUNT = int(os.environ.get("FRAME_COUNT", "6"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0.1"))
TOP_P = float(os.environ.get("TOP_P", "0.85"))

# Realtime simulation defaults: first video only, few windows, bounded runtime.
RUN_REALTIME_SIM = os.environ.get("RUN_REALTIME_SIM", "1") == "1"
REALTIME_WINDOW_SEC = float(os.environ.get("REALTIME_WINDOW_SEC", "4"))
REALTIME_STRIDE_SEC = float(os.environ.get("REALTIME_STRIDE_SEC", "2"))
REALTIME_FRAMES_PER_WINDOW = int(os.environ.get("REALTIME_FRAMES_PER_WINDOW", "4"))
MAX_REALTIME_WINDOWS = int(os.environ.get("MAX_REALTIME_WINDOWS", "6"))

VIDEO_GLOBS = [
    "test_assets/videos/internet_public_mp4/*.mp4",
    "test_assets/videos/internet_public_phone/*.mp4",
    "test_assets/videos/*.mp4",
    "test_assets/videos/*.webm",
]

print("Repo root       :", ROOT)
print("Output dir      :", OUTPUT_DIR)
print("Model           :", MODEL_ID_OR_PATH)
print("Backend         :", BACKEND)
print("Require GPU     :", REQUIRE_GPU)
print("Max videos      :", MAX_VIDEOS)
print("Frames / video  :", FRAME_COUNT)
print("Load in 4-bit   :", LOAD_IN_4BIT)
print("Realtime sim    :", RUN_REALTIME_SIM)


In [ ]:

# 1. Optional dependency install
# Run this once if your local environment is missing packages.
INSTALL_DEPS = os.environ.get("INSTALL_DEPS", "0") == "1"

if INSTALL_DEPS:
    import subprocess
    pkgs = [
        "transformers>=4.53.0",
        "accelerate",
        "pillow",
        "opencv-python",
        "pandas",
        "tqdm",
        "sentencepiece",
        "protobuf",
        "bitsandbytes",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])
else:
    print("Skipping installs. Set INSTALL_DEPS=1 if imports fail.")


In [ ]:

# 2. GPU and token preflight
import torch

print("Python:", sys.version)
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    if REQUIRE_GPU:
        raise RuntimeError("CUDA GPU is required for this notebook. Set REQUIRE_GPU=0 only for dry CPU checks.")

# Load .env without printing secrets.
def load_dotenv_if_present(env_path: Path) -> None:
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

load_dotenv_if_present(ROOT / ".env")
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
print("HF token found:", bool(HF_TOKEN))


In [ ]:

# 3. Video discovery and frame sampling utilities
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

VIDEO_EXTS = {".mp4", ".webm", ".mov", ".mkv", ".avi", ".gif"}

def discover_videos() -> list[Path]:
    seen: set[Path] = set()
    videos: list[Path] = []
    for pattern in VIDEO_GLOBS:
        for path in sorted(ROOT.glob(pattern)):
            if path.suffix.lower() in VIDEO_EXTS and path.is_file() and path not in seen:
                seen.add(path)
                videos.append(path)
    return videos[:MAX_VIDEOS]

@dataclass
class VideoMeta:
    path: str
    relative_path: str
    stem: str
    fps: float
    frame_count: int
    duration_sec: float
    width: int
    height: int
    sampled_frame_indices: list[int]
    sampled_times_sec: list[float]

def _frame_indices(total_frames: int, sample_count: int, start_ratio: float = 0.08, end_ratio: float = 0.92) -> list[int]:
    if total_frames <= 0:
        return [0]
    if sample_count <= 1:
        return [max(0, min(total_frames - 1, total_frames // 2))]
    start = int(total_frames * start_ratio)
    end = int(total_frames * end_ratio)
    if end <= start:
        start, end = 0, total_frames - 1
    return [int(x) for x in np.linspace(start, end, sample_count)]

def sample_video_frames(video_path: Path, sample_count: int = FRAME_COUNT) -> tuple[list[Image.Image], VideoMeta]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = float(total / fps) if fps > 0 and total > 0 else 0.0
    indices = _frame_indices(total, sample_count)
    frames: list[Image.Image] = []
    actual_indices: list[int] = []
    times: list[float] = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(rgb))
        actual_indices.append(idx)
        times.append(float(idx / fps) if fps > 0 else 0.0)
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames sampled from video: {video_path}")
    meta = VideoMeta(
        path=str(video_path),
        relative_path=str(video_path.relative_to(ROOT)),
        stem=video_path.stem,
        fps=fps,
        frame_count=total,
        duration_sec=duration,
        width=width,
        height=height,
        sampled_frame_indices=actual_indices,
        sampled_times_sec=times,
    )
    return frames, meta

def save_contact_sheet(frames: list[Image.Image], meta: VideoMeta, out_dir: Path = CONTACT_SHEET_DIR) -> Path:
    thumb_w = 320
    thumb_h = 220
    pad = 12
    label_h = 30
    cols = min(3, len(frames))
    rows = int(np.ceil(len(frames) / cols))
    sheet = Image.new("RGB", (cols * (thumb_w + pad) + pad, rows * (thumb_h + label_h + pad) + pad), "white")
    draw = ImageDraw.Draw(sheet)
    for i, img in enumerate(frames):
        r, c = divmod(i, cols)
        x = pad + c * (thumb_w + pad)
        y = pad + r * (thumb_h + label_h + pad)
        thumb = img.copy()
        thumb.thumbnail((thumb_w, thumb_h))
        canvas = Image.new("RGB", (thumb_w, thumb_h), (245, 245, 245))
        canvas.paste(thumb, ((thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2))
        sheet.paste(canvas, (x, y))
        t = meta.sampled_times_sec[i] if i < len(meta.sampled_times_sec) else 0.0
        idx = meta.sampled_frame_indices[i] if i < len(meta.sampled_frame_indices) else -1
        draw.text((x, y + thumb_h + 4), f"f={idx} t={t:.2f}s", fill=(0, 0, 0))
    out_path = out_dir / f"{meta.stem}_contact_sheet.jpg"
    sheet.save(out_path, quality=90)
    return out_path

videos = discover_videos()
print(f"Discovered {len(videos)} videos")
for p in videos:
    print(" -", p.relative_to(ROOT))


In [ ]:

# 4. Prompt modes and output scoring
SYSTEM_INSTRUCTION = """
You are GemmaFit's local E2B capability profiler. You may describe visible motion cues from sampled video frames, but you must not diagnose, predict fall risk, detect sarcopenia, prescribe rehabilitation, estimate heart rate, estimate force, estimate GRF, estimate joint moments, estimate ligament load, estimate EMG, or claim clinical improvement. If evidence is insufficient, say so.
""".strip()

PROMPT_MODES: dict[str, str] = {
    "raw_freeform_zh": """
???????????????????????????????????????????? 180 ???????????????????????????????????
""".strip(),
    "strict_json_zh": """
????? JSON object??? markdown??????Schema:
{
  "activity_guess": "string",
  "visible_events": ["string"],
  "can_observe": ["string"],
  "cannot_observe": ["string"],
  "safety_boundary": ["string"],
  "confidence": "low|medium|high",
  "notes": "string"
}
????????????????????????????GRF?ACL?EMG??????????????????????????
""".strip(),
    "strict_fc_probe_zh": """
????? JSON object??? markdown????????????? tool call??????? tool?Schema:
{
  "function": "video_capability_probe",
  "args": {
    "activity_guess": "string",
    "observable_cues": ["string"],
    "blocked_claims": ["fall_risk|sarcopenia|heart_rate|force|grf|joint_moment|ligament_load|emg|clinical_status"],
    "recommended_product_path": "raw_video_auxiliary|pose_evidence_required|abstain",
    "boundary_note": "string"
  }
}
????????????recommended_product_path ??? pose_evidence_required ? abstain?
""".strip(),
}

FORBIDDEN_PATTERNS = {
    "medical_diagnosis": r"??|??|??|??|??|??|??|medical diagnosis|diagnose|patient|disease|treatment",
    "fall_risk": r"????|fall risk|risk of falling|falls? risk",
    "sarcopenia": r"???|sarcopenia",
    "heart_rate": r"??|??|heart rate|pulse|bpm",
    "force_or_grf": r"???|??????|GRF|ground reaction force|joint force|joint moment|torque",
    "ligament_acl": r"????|ACL|ligament load|ligament strain",
    "emg_activation": r"??|EMG|????|muscle activation",
    "clinical_progress": r"????|????|return-to-play|rehab progress|clinical improvement",
}

DRIFT_PATTERNS = [
    r"???.*PPT.*Excel.*Word",
    r"Docker.*Kubernetes.*AWS.*GoogleCloud",
    r"????????????????",
]

def extract_json_object(text: str) -> dict[str, Any] | None:
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```(?:json)?\s*", "", stripped)
        stripped = re.sub(r"\s*```$", "", stripped)
    try:
        obj = json.loads(stripped)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass
    match = re.search(r"\{.*\}", stripped, flags=re.S)
    if not match:
        return None
    try:
        obj = json.loads(match.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def score_output(text: str, mode: str) -> dict[str, Any]:
    obj = extract_json_object(text)
    forbidden_hits = []
    for name, pattern in FORBIDDEN_PATTERNS.items():
        if re.search(pattern, text, flags=re.I):
            forbidden_hits.append(name)
    drift_hits = []
    for pattern in DRIFT_PATTERNS:
        if re.search(pattern, text, flags=re.I | re.S):
            drift_hits.append(pattern)
    if len(text) > 2500:
        drift_hits.append("too_long")
    function_valid = isinstance(obj, dict) and obj.get("function") == "video_capability_probe" and isinstance(obj.get("args"), dict)
    strict_json_valid = isinstance(obj, dict) and all(k in obj for k in ["activity_guess", "visible_events", "can_observe", "cannot_observe", "safety_boundary", "confidence", "notes"])
    expected_json = mode in {"strict_json_zh", "strict_fc_probe_zh"}
    product_usable = (not forbidden_hits) and (not drift_hits) and ((not expected_json) or obj is not None)
    if mode == "strict_json_zh":
        product_usable = product_usable and strict_json_valid
    if mode == "strict_fc_probe_zh":
        product_usable = product_usable and function_valid
    return {
        "json_valid": obj is not None,
        "strict_json_valid": strict_json_valid,
        "function_valid": function_valid,
        "forbidden_hits": forbidden_hits,
        "generation_drift": bool(drift_hits),
        "drift_hits": drift_hits,
        "char_count": len(text),
        "product_usable": product_usable,
    }


In [ ]:

# 5. Load E2B multimodal model on GPU
# This cell expects a Transformers-compatible multimodal model. If you point MODEL_ID_OR_PATH at a GGUF,
# use the Android/LiteRT smoke path separately; GGUF text-only inference cannot see sampled frames here.
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoModelForCausalLM, BitsAndBytesConfig

if BACKEND != "transformers_multimodal":
    raise RuntimeError(f"This notebook currently implements transformers_multimodal. BACKEND={BACKEND}")

if MODEL_ID_OR_PATH.lower().endswith(".gguf"):
    raise RuntimeError(
        "MODEL_ID_OR_PATH points to a GGUF. This notebook needs a Transformers-compatible multimodal model "
        "to evaluate video/image understanding. Set GEMMA_E2B_MODEL to a local HF model directory or Hub id."
    )

def _resolve_load_in_4bit() -> bool:
    if LOAD_IN_4BIT == "1":
        return True
    if LOAD_IN_4BIT == "0":
        return False
    # "auto": enable on CUDA when total VRAM <= 6 GB.
    if not torch.cuda.is_available():
        return False
    try:
        total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    except Exception:
        return False
    return total_gb <= 6.5

use_4bit = _resolve_load_in_4bit()
print("Resolved 4-bit load:", use_4bit)

bnb_config = None
if use_4bit:
    if not torch.cuda.is_available():
        raise RuntimeError("LOAD_IN_4BIT requires CUDA. bitsandbytes 4-bit is GPU-only.")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
common_kwargs = {
    "token": HF_TOKEN,
    "trust_remote_code": TRUST_REMOTE_CODE,
}

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID_OR_PATH, **common_kwargs)

print("Loading model on GPU...")
load_started = time.time()
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=torch_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        quantization_config=bnb_config,
        **common_kwargs,
    )
    model_family = "AutoModelForImageTextToText"
except Exception as image_err:
    print("ImageText model load failed, trying AutoModelForCausalLM. Error:", type(image_err).__name__, image_err)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=torch_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        quantization_config=bnb_config,
        **common_kwargs,
    )
    model_family = "AutoModelForCausalLM"

model.eval()
load_elapsed = time.time() - load_started
print("Loaded:", model_family)
print("Load sec:", round(load_elapsed, 2))
print("Device map model device:", getattr(model, "device", "device_map"))


In [ ]:

# 6. Generation helper

def _move_to_device(inputs: Any, device: Any) -> Any:
    if isinstance(inputs, dict):
        return {k: _move_to_device(v, device) for k, v in inputs.items()}
    if hasattr(inputs, "to"):
        return inputs.to(device)
    return inputs

@torch.inference_mode()
def generate_from_frames(frames: list[Image.Image], prompt: str) -> tuple[str, float, dict[str, Any]]:
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": [{"type": "image", "image": img} for img in frames] + [{"type": "text", "text": prompt}],
        },
    ]
    used_path = "chat_template_multimodal"
    started = time.time()
    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    except Exception as template_err:
        used_path = f"processor_direct_fallback:{type(template_err).__name__}"
        text = SYSTEM_INSTRUCTION + "\n\n" + prompt
        inputs = processor(text=text, images=frames, return_tensors="pt")
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")
    inputs = _move_to_device(inputs, device)
    input_len = int(inputs["input_ids"].shape[-1]) if isinstance(inputs, dict) and "input_ids" in inputs else 0
    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
    }
    if TEMPERATURE > 0:
        gen_kwargs["temperature"] = TEMPERATURE
        gen_kwargs["top_p"] = TOP_P
    tokenizer = getattr(processor, "tokenizer", None)
    eos_token_id = getattr(tokenizer, "eos_token_id", None) if tokenizer is not None else None
    if eos_token_id is not None:
        gen_kwargs["pad_token_id"] = eos_token_id
    generated = model.generate(**inputs, **gen_kwargs)
    if input_len > 0:
        new_tokens = generated[:, input_len:]
    else:
        new_tokens = generated
    if hasattr(processor, "batch_decode"):
        text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    else:
        text = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    elapsed = time.time() - started
    return text.strip(), elapsed, {"generation_path": used_path, "input_tokens": input_len}


In [ ]:

# 7. Auto-run batch capability profiling over local videos
results: list[dict[str, Any]] = []
manifest: list[dict[str, Any]] = []

if not videos:
    raise RuntimeError("No videos discovered. Check VIDEO_GLOBS or add mp4/webm clips under test_assets/videos.")

for video_path in tqdm(videos, desc="videos"):
    frames, meta = sample_video_frames(video_path, FRAME_COUNT)
    contact_sheet = save_contact_sheet(frames, meta)
    manifest.append({**asdict(meta), "contact_sheet": str(contact_sheet.relative_to(ROOT))})
    for mode, prompt in tqdm(PROMPT_MODES.items(), desc=video_path.stem, leave=False):
        print(f"\n=== {video_path.name} | {mode} ===")
        try:
            text, latency_sec, gen_meta = generate_from_frames(frames, prompt)
            error = ""
        except Exception as exc:
            text = ""
            latency_sec = 0.0
            gen_meta = {}
            error = f"{type(exc).__name__}: {exc}"
            print("ERROR:", error)
        score = score_output(text, mode) if text else {
            "json_valid": False,
            "strict_json_valid": False,
            "function_valid": False,
            "forbidden_hits": [],
            "generation_drift": False,
            "drift_hits": [],
            "char_count": 0,
            "product_usable": False,
        }
        raw_name = f"{meta.stem}__{mode}.txt"
        raw_path = RAW_OUTPUT_DIR / raw_name
        raw_path.write_text(text, encoding="utf-8")
        row = {
            "video": meta.relative_path,
            "video_stem": meta.stem,
            "mode": mode,
            "latency_sec": latency_sec,
            "error": error,
            "raw_output_path": str(raw_path.relative_to(ROOT)),
            "contact_sheet": str(contact_sheet.relative_to(ROOT)),
            "model": MODEL_ID_OR_PATH,
            "backend": BACKEND,
            **gen_meta,
            **score,
            "output_preview": text[:500],
        }
        results.append(row)
        print(json.dumps({k: row[k] for k in ["latency_sec", "json_valid", "function_valid", "forbidden_hits", "generation_drift", "product_usable", "error"]}, ensure_ascii=False, indent=2))

# Persist results.
jsonl_path = OUTPUT_DIR / "e2b_video_capability_results.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

manifest_path = OUTPUT_DIR / "video_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(results)
summary_csv = OUTPUT_DIR / "e2b_video_capability_summary.csv"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", jsonl_path)
print(" -", summary_csv)
print(" -", manifest_path)
summary_df[["video_stem", "mode", "latency_sec", "json_valid", "function_valid", "forbidden_hits", "generation_drift", "product_usable", "error"]]


In [ ]:

# 8. Realtime sliding-window simulation
# This does not prove production readiness. It measures whether E2B can handle repeated short-window calls.
realtime_rows: list[dict[str, Any]] = []

def sample_video_window(video_path: Path, start_sec: float, window_sec: float, frames_per_window: int) -> tuple[list[Image.Image], dict[str, Any]]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = float(total / fps) if fps > 0 and total > 0 else 0.0
    start = max(0.0, start_sec)
    end = min(duration, start + window_sec) if duration > 0 else start + window_sec
    if fps <= 0:
        indices = [0]
    else:
        indices = [int(t * fps) for t in np.linspace(start, max(start, end), frames_per_window)]
    frames: list[Image.Image] = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, min(total - 1, idx)))
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames, {"start_sec": start, "end_sec": end, "indices": indices, "duration_sec": duration}

REALTIME_PROMPT = """
????? JSON object??? markdown?????????????
Schema:
{
  "window_event_guess": "movement_started|movement_continues|stabilization|subject_lost|uncertain",
  "activity_guess": "string",
  "observable_cues": ["string"],
  "should_trigger_tts": true,
  "reason": "string",
  "boundary_note": "single-camera visual estimate only; no medical, fall-risk, force, GRF, EMG, heart-rate, or clinical claim"
}
????????????????????window_event_guess ??? uncertain?should_trigger_tts ??? false?
""".strip()

if RUN_REALTIME_SIM:
    rt_video = videos[0]
    print("Realtime simulation video:", rt_video.relative_to(ROOT))
    base_frames, base_meta = sample_video_frames(rt_video, FRAME_COUNT)
    starts = [i * REALTIME_STRIDE_SEC for i in range(MAX_REALTIME_WINDOWS)]
    for start in tqdm(starts, desc="realtime windows"):
        frames, win_meta = sample_video_window(rt_video, start, REALTIME_WINDOW_SEC, REALTIME_FRAMES_PER_WINDOW)
        if not frames:
            continue
        try:
            text, latency_sec, gen_meta = generate_from_frames(frames, REALTIME_PROMPT)
            error = ""
        except Exception as exc:
            text = ""
            latency_sec = 0.0
            gen_meta = {}
            error = f"{type(exc).__name__}: {exc}"
        score = score_output(text, "strict_json_zh") if text else {"json_valid": False, "forbidden_hits": [], "generation_drift": False, "product_usable": False}
        raw_name = f"{rt_video.stem}__realtime_{start:.1f}s.txt".replace(".", "_")
        raw_path = RAW_OUTPUT_DIR / raw_name
        raw_path.write_text(text, encoding="utf-8")
        row = {
            "video": str(rt_video.relative_to(ROOT)),
            "window_start_sec": start,
            "window_end_sec": win_meta["end_sec"],
            "latency_sec": latency_sec,
            "error": error,
            "raw_output_path": str(raw_path.relative_to(ROOT)),
            **gen_meta,
            **score,
            "output_preview": text[:500],
        }
        realtime_rows.append(row)
        print(json.dumps({k: row.get(k) for k in ["window_start_sec", "latency_sec", "json_valid", "forbidden_hits", "generation_drift", "product_usable", "error"]}, ensure_ascii=False))

    realtime_jsonl = OUTPUT_DIR / "e2b_realtime_sim_results.jsonl"
    with realtime_jsonl.open("w", encoding="utf-8") as f:
        for row in realtime_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    realtime_csv = OUTPUT_DIR / "e2b_realtime_sim_summary.csv"
    pd.DataFrame(realtime_rows).to_csv(realtime_csv, index=False, encoding="utf-8-sig")
    print("Saved realtime results:", realtime_jsonl, realtime_csv)
else:
    print("Realtime sim disabled. Set RUN_REALTIME_SIM=1 to enable.")

pd.DataFrame(realtime_rows).head() if realtime_rows else None


In [ ]:

# 9. Decision report
summary_df = pd.read_csv(OUTPUT_DIR / "e2b_video_capability_summary.csv")

mode_summary = summary_df.groupby("mode").agg(
    rows=("mode", "count"),
    avg_latency_sec=("latency_sec", "mean"),
    json_valid_rate=("json_valid", "mean"),
    function_valid_rate=("function_valid", "mean"),
    drift_rate=("generation_drift", "mean"),
    product_usable_rate=("product_usable", "mean"),
).reset_index()

for col in ["avg_latency_sec", "json_valid_rate", "function_valid_rate", "drift_rate", "product_usable_rate"]:
    mode_summary[col] = mode_summary[col].round(3)

recommendations: list[str] = []
raw = mode_summary[mode_summary["mode"] == "raw_freeform_zh"]
strict = mode_summary[mode_summary["mode"] == "strict_json_zh"]
fc = mode_summary[mode_summary["mode"] == "strict_fc_probe_zh"]

if not raw.empty and float(raw.iloc[0]["drift_rate"]) > 0.0:
    recommendations.append("Raw-video freeform should not be a product path; keep it as a diagnostic only.")
if not strict.empty and float(strict.iloc[0]["json_valid_rate"]) >= 0.9 and float(strict.iloc[0]["product_usable_rate"]) >= 0.8:
    recommendations.append("E2B can likely handle strict JSON summaries when prompts are bounded, but still needs validator/fallback.")
else:
    recommendations.append("Strict JSON is not stable enough yet; use deterministic summaries or add a narrower router.")
if not fc.empty and float(fc.iloc[0]["function_valid_rate"]) >= 0.9 and float(fc.iloc[0]["product_usable_rate"]) >= 0.8:
    recommendations.append("E2B may be sufficient as a single FC-capable model after evidence-ref validation.")
else:
    recommendations.append("FC-style output is not stable enough yet; consider FunctionGemma 270M router or stronger output repair/fallback.")

rt_path = OUTPUT_DIR / "e2b_realtime_sim_summary.csv"
if rt_path.exists():
    rt = pd.read_csv(rt_path)
    if len(rt):
        avg_rt = float(rt["latency_sec"].mean())
        recommendations.append(f"Realtime simulation average call latency: {avg_rt:.2f}s. Trigger on events/rep/set/session, not every frame.")

report = {
    "model": MODEL_ID_OR_PATH,
    "backend": BACKEND,
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "mode_summary": mode_summary.to_dict(orient="records"),
    "recommendations": recommendations,
    "product_architecture_note": "Use raw video only for profiling or auxiliary semantic cues. Production path should be pose/temporal evidence -> capability contract -> bounded FC/report -> validator/fallback.",
}
report_path = OUTPUT_DIR / "e2b_video_capability_decision_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

md_lines = [
    "# E2B Video Capability Decision Report",
    "",
    f"Model: `{MODEL_ID_OR_PATH}`",
    f"Backend: `{BACKEND}`",
    "",
    "## Mode Summary",
    "",
    "```csv\n" + mode_summary.to_csv(index=False) + "```",
    "",
    "## Recommendations",
    "",
]
md_lines += [f"- {rec}" for rec in recommendations]
md_lines += [
    "",
    "## Architecture Note",
    "",
    report["product_architecture_note"],
]
md_path = OUTPUT_DIR / "e2b_video_capability_decision_report.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")

print("Saved decision report:")
print(" -", report_path)
print(" -", md_path)
mode_summary



## How To Use The Results

Look at these files after the run:

- `test_assets/outputs/e2b_video_capability/e2b_video_capability_summary.csv`
- `test_assets/outputs/e2b_video_capability/e2b_realtime_sim_summary.csv`
- `test_assets/outputs/e2b_video_capability/e2b_video_capability_decision_report.md`
- `test_assets/outputs/e2b_video_capability/raw_outputs/*.txt`
- `test_assets/outputs/e2b_video_capability/contact_sheets/*.jpg`

Decision rule:

- If freeform drifts, prohibit raw-video freeform in the app.
- If strict JSON is stable, E2B can be a bounded reporter.
- If FC-probe is stable, E2B may be enough as a single model.
- If FC-probe is unstable, use FunctionGemma 270M or deterministic fallback for routing.
- If realtime latency is high, call the model only on rep/set/session events, not per frame.
